In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import gc
import os
import sys

import warnings
warnings.filterwarnings('ignore')

In [ ]:
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

In [ ]:
import shap
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

In [ ]:
## load imp genes and their Entrez Gene ID

imp_gene_entrez_ID = pd.read_csv("Z:/multiomics based manuscript/REVISION AFTER JMS/OUP_BA_REVIEWER_COMMENTS/independent_cohort/imp_genes_entrez_ID.csv")

In [ ]:
imp_entrez_ID = list(imp_gene_entrez_ID['EntrezID'])

In [ ]:
imp_gene_entrez_ID

In [ ]:
df_luad_rna = pd.read_csv("Z:/multiomics based manuscript/REVISION AFTER JMS/OUP_BA_REVIEWER_COMMENTS/independent_cohort/CPTAC_GDC/luad_cptac_gdc/luad_cptac_gdc/data_mrna_seq_fpkm_zscores_ref_all_samples.txt", sep='\t')
df_luad_rna.index=df_luad_rna['Entrez_Gene_Id']
df_luad_rna.drop(columns=['Entrez_Gene_Id'], inplace=True)
df_luad_rna = df_luad_rna.T

In [ ]:
df_luad_cnv = pd.read_csv("Z:/multiomics based manuscript/REVISION AFTER JMS/OUP_BA_REVIEWER_COMMENTS/independent_cohort/CPTAC_GDC/luad_cptac_gdc/luad_cptac_gdc/data_cna.txt", sep='\t')
df_luad_cnv.index=df_luad_cnv['Entrez_Gene_Id']
df_luad_cnv.drop(columns=['Entrez_Gene_Id'], inplace=True)
df_luad_cnv = df_luad_cnv.T

In [ ]:
df_lusc_rna = pd.read_csv("Z:/multiomics based manuscript/REVISION AFTER JMS/OUP_BA_REVIEWER_COMMENTS/independent_cohort/CPTAC_GDC/lusc_cptac_gdc/lusc_cptac_gdc/data_mrna_seq_fpkm_zscores_ref_all_samples.txt", sep='\t')
df_lusc_rna.index=df_lusc_rna['Entrez_Gene_Id']
df_lusc_rna.drop(columns=['Entrez_Gene_Id'], inplace=True)
df_lusc_rna = df_lusc_rna.T

In [ ]:
df_lusc_cnv = pd.read_csv("Z:/multiomics based manuscript/REVISION AFTER JMS/OUP_BA_REVIEWER_COMMENTS/independent_cohort/CPTAC_GDC/lusc_cptac_gdc/lusc_cptac_gdc/data_cna.txt", sep='\t')
df_lusc_cnv.index=df_lusc_cnv['Entrez_Gene_Id']
df_lusc_cnv.drop(columns=['Entrez_Gene_Id'], inplace=True)
df_lusc_cnv = df_lusc_cnv.T

In [ ]:
## df.columns.intersection keeps only imp_gene list values as columns

df_lusc_rna = df_lusc_rna[df_lusc_rna.columns.intersection(imp_entrez_ID)]
df_lusc_cnv = df_lusc_cnv[df_lusc_cnv.columns.intersection(imp_entrez_ID)]

df_luad_rna = df_luad_rna[df_luad_rna.columns.intersection(imp_entrez_ID)]
df_luad_cnv = df_luad_cnv[df_luad_cnv.columns.intersection(imp_entrez_ID)]


In [ ]:
common_samples = df_luad_rna.index.intersection(df_luad_cnv.index)
# df_luad = pd.concat([df_luad_rna.loc[common_samples], df_luad_cnv.loc[common_samples]])

In [ ]:
len(common_samples)

In [ ]:
common_df_luad_rna = df_luad_rna.loc[common_samples]
common_df_luad_cnv = df_luad_cnv.loc[common_samples]

In [ ]:
common_samples = df_lusc_rna.index.intersection(df_lusc_cnv.index)

In [ ]:
len(common_samples)

In [ ]:
common_df_lusc_rna = df_lusc_rna.loc[common_samples]
common_df_lusc_cnv = df_lusc_cnv.loc[common_samples]

In [ ]:
common_df_luad_rna['label'] = 1
common_df_lusc_rna['label'] = 0
df_rna = pd.concat([common_df_luad_rna, common_df_lusc_rna], axis=0)

common_df_luad_cnv['label'] = 1
common_df_lusc_cnv['label'] = 0
df_cnv = pd.concat([common_df_luad_cnv, common_df_lusc_cnv], axis=0)

In [ ]:
df_full = pd.concat([df_rna, df_cnv], axis=0)

In [ ]:
df_full

In [ ]:
## create a mapping of entrezGeneID with their name
mapping_list = []
for ID in df_full.columns:
    if ID == 'label':
        continue
    else:
        mapping_list.append(imp_gene_entrez_ID.loc[imp_gene_entrez_ID['EntrezID']==ID,'GeneName'].values[0])

In [ ]:
mapping_list.append('label')

In [ ]:
df_rna.columns = mapping_list
df_cnv.columns = mapping_list
df_full.columns = mapping_list

In [ ]:
df_rna.reset_index(drop=True, inplace=True)
df_cnv.reset_index(drop=True, inplace=True)
df_full.reset_index(drop=True, inplace=True)

In [ ]:
## perform classification using XGBoost and SHAP

def perform_classification_and_SHAP(df, random_state):
    
    
    # Split X and y
    X = df.drop(columns=["label"])
    y = df["label"]

    feature_names = X.columns

    # Train-test split (80% train, 20% validation)
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, stratify=y, random_state=random_state)

    # Train the XGBoost model on training split only
    model = XGBClassifier(device='cuda')
    model.fit(X_train, y_train)

    # Compute SHAP values on the validation set only
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_val)

    # SHAP bar plot – mean absolute SHAP values (on validation)
    shap.summary_plot(
        shap_values,
        X_val,
        feature_names=feature_names,
        plot_type="bar"
    )

    return shap_values, feature_names

In [ ]:
for seed in range(1, 6):

     ## sample with same seed, else samples would shuffle and then fusing of probs will not be correct
    _df_rna_xena = df_rna.sample(frac=1, replace=False, ignore_index=True, random_state = seed)
    _df_cnv_xena = df_cnv.sample(frac=1, replace=False, ignore_index=True, random_state = seed)
    _df_full_xena = df_full.sample(frac=1, replace=False, ignore_index=True, random_state = seed)
    
    
    ## get XGBoost results on RNASeq
    results_rna, feats = perform_classification_and_SHAP(_df_rna_xena, seed)
    results_df = pd.DataFrame(results_rna, columns=feats)
    results_df.to_csv(f"Z:/multiomics based manuscript/REVISION AFTER JMS/OUP_BA_REVIEWER_COMMENTS/independent_cohort/RESULTS/SHAP/results_rna_{seed}.csv", index=False) ## Print results    

    ## get XGBoost results on CNV
    results_cnv, feats = perform_classification_and_SHAP(_df_cnv_xena, seed)
    results_df = pd.DataFrame(results_cnv, columns=feats)
    results_df.to_csv(f"Z:/multiomics based manuscript/REVISION AFTER JMS/OUP_BA_REVIEWER_COMMENTS/independent_cohort/RESULTS/SHAP/results_cnv_{seed}.csv", index=False) ## Print results

    ## get XGBoost results on full
    results_full, feats = perform_classification_and_SHAP(_df_full_xena, seed)
    results_df = pd.DataFrame(results_full, columns=feats)
    results_df.to_csv(f"Z:/multiomics based manuscript/REVISION AFTER JMS/OUP_BA_REVIEWER_COMMENTS/independent_cohort/RESULTS/SHAP/results_full_{seed}.csv", index=False) ## Print results

    
    print("--------------------------------------------------------------------------------")
    